# Mixture of Gaussians Model

## Learning Objectives
- Understand the **mixture of Gaussians (GMM)** as a generative model with latent variables
- Define the joint distribution $p(x^{(i)}, z^{(i)}) = p(x^{(i)} | z^{(i)}) p(z^{(i)})$
- Understand why maximising the log-likelihood with latent variables has no closed-form solution
- See that if the $z^{(i)}$'s were known, MLE reduces to Gaussian discriminant analysis
- Contrast GMM (**soft** cluster assignments) with k-means (**hard** assignments)

## Problem Statement

We wish to model an unlabelled dataset $\{x^{(1)}, \ldots, x^{(m)}\}$ as arising from $k$ Gaussians.

**Generative model:**
1. Draw cluster identity: $z^{(i)} \sim \text{Multinomial}(\phi)$, where $\phi_j = P(z^{(i)} = j)$, $\sum_j \phi_j = 1$
2. Draw observation: $x^{(i)} | z^{(i)} = j \sim \mathcal{N}(\mu_j, \Sigma_j)$

Parameters: $\phi \in \mathbb{R}^k$, $\{\mu_j\}_{j=1}^k$, $\{\Sigma_j\}_{j=1}^k$.

The $z^{(i)}$'s are **latent** — hidden, unobserved.

**Log-likelihood (marginalising over $z$):**
$$\ell(\phi, \mu, \Sigma) = \sum_{i=1}^m \log \sum_{z^{(i)}=1}^k p(x^{(i)} | z^{(i)}; \mu, \Sigma)\, p(z^{(i)}; \phi)$$

Setting derivatives to zero gives no closed-form solution — the $\log\sum$ cannot be simplified.

## 1. The Generative Story

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

np.random.seed(42)

# True GMM parameters
k    = 3
phi  = np.array([0.4, 0.35, 0.25])
mus  = np.array([[0.0, 0.0], [4.0, 1.0], [2.0, 4.5]])
sigs = [
    np.array([[1.0, 0.5], [0.5, 0.8]]),
    np.array([[0.8, -0.3], [-0.3, 1.2]]),
    np.array([[1.5, 0.0], [0.0, 0.5]]),
]
m = 200

# Sample from the GMM
z_true = np.random.choice(k, size=m, p=phi)
X = np.array([np.random.multivariate_normal(mus[z], sigs[z]) for z in z_true])

colors = ['#2166ac', '#d6604d', '#4dac26']
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: observed data (no labels)
ax = axes[0]
ax.scatter(X[:, 0], X[:, 1], c='gray', s=25, alpha=0.6)
ax.set_title('Observed data $\\{x^{(i)}\\}$ — no labels\n'
             '(unsupervised; latent $z^{(i)}$ not observed)', fontsize=10)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')

# Right: true generative structure revealed
ax = axes[1]
xg = np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 200)
yg = np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 200)
Xg, Yg = np.meshgrid(xg, yg)
pos = np.dstack([Xg, Yg])

for j in range(k):
    pts = X[z_true == j]
    ax.scatter(pts[:, 0], pts[:, 1], c=colors[j], s=25, alpha=0.6,
               label=f'$z={j+1}$, $\\phi_{j+1}={phi[j]:.2f}$')
    ax.scatter(*mus[j], marker='X', s=220, c=colors[j], edgecolors='k', lw=1.5, zorder=5)
    Z_pdf = multivariate_normal(mus[j], sigs[j]).pdf(pos)
    ax.contour(Xg, Yg, Z_pdf, levels=4, colors=[colors[j]], alpha=0.5, linewidths=1.5)

ax.set_title('True GMM structure ($z^{(i)}$ revealed)\n✕ = true means $\\mu_j$', fontsize=10)
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
ax.legend(fontsize=9)

fig.suptitle('Mixture of Gaussians — Generative Story', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. If $z^{(i)}$ Were Known: Closed-Form MLE

If we knew the cluster assignments $z^{(i)}$ for each point, the complete-data log-likelihood is:
$$\ell(\phi, \mu, \Sigma) = \sum_{i=1}^m \log p(x^{(i)} | z^{(i)}; \mu, \Sigma) + \log p(z^{(i)}; \phi)$$

Maximising over $\phi$, $\mu_j$, $\Sigma_j$ gives closed-form solutions:
$$\phi_j = \frac{1}{m}\sum_{i=1}^m \mathbf{1}\{z^{(i)}=j\}, \quad
\mu_j = \frac{\sum_i \mathbf{1}\{z^{(i)}=j\} x^{(i)}}{\sum_i \mathbf{1}\{z^{(i)}=j\}}, \quad
\Sigma_j = \frac{\sum_i \mathbf{1}\{z^{(i)}=j\}(x^{(i)}-\mu_j)(x^{(i)}-\mu_j)^T}{\sum_i \mathbf{1}\{z^{(i)}=j\}}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

def mle_with_labels(X, z, k):
    m = len(X)
    phi_est = np.array([np.mean(z == j) for j in range(k)])
    mu_est  = np.array([X[z == j].mean(axis=0) for j in range(k)])
    sig_est = []
    for j in range(k):
        diff = X[z == j] - mu_est[j]
        sig_est.append(diff.T @ diff / (z == j).sum())
    return phi_est, mu_est, sig_est

phi_est, mu_est, sig_est = mle_with_labels(X, z_true, k)

print('True vs. MLE parameters (with known z):')
for j in range(k):
    print(f'  Cluster {j+1}: phi true={phi[j]:.2f} est={phi_est[j]:.2f} | '
          f'mu true={mus[j]} est={mu_est[j].round(2)}')

# Plot estimated contours vs true contours
fig, ax = plt.subplots(figsize=(8, 6))
xg = np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 200)
yg = np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 200)
Xg, Yg = np.meshgrid(xg, yg); pos = np.dstack([Xg, Yg])

for j in range(k):
    pts = X[z_true == j]
    ax.scatter(pts[:, 0], pts[:, 1], c=colors[j], s=20, alpha=0.5)
    Z_true = multivariate_normal(mus[j],    sigs[j]   ).pdf(pos)
    Z_est  = multivariate_normal(mu_est[j], sig_est[j]).pdf(pos)
    ax.contour(Xg, Yg, Z_true, levels=3, colors=[colors[j]], linewidths=2,   linestyles='-',  alpha=0.8)
    ax.contour(Xg, Yg, Z_est,  levels=3, colors=[colors[j]], linewidths=1.5, linestyles='--', alpha=0.8)

from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0],[0], color='gray', lw=2,   ls='-',  label='True Gaussians'),
    Line2D([0],[0], color='gray', lw=1.5, ls='--', label='MLE estimate (known $z$)'),
], fontsize=9)
ax.set_title('MLE with known $z^{(i)}$ closely recovers the true Gaussians\n'
             '(solid = true, dashed = MLE estimate)')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
plt.tight_layout()
plt.show()

## 3. Why the Marginal Likelihood Is Hard

When $z^{(i)}$ is unobserved, we marginalise:
$$p(x^{(i)}; \phi, \mu, \Sigma) = \sum_{j=1}^k p(x^{(i)} | z^{(i)}=j; \mu, \Sigma)\, \phi_j$$

The log-likelihood becomes $\ell = \sum_i \log \sum_j (\ldots)$ — a **log of a sum**.
Setting $\nabla_{\mu_j} \ell = 0$ gives implicit equations with no closed-form solution.
This is exactly why we need the EM algorithm.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

def gmm_log_likelihood(X, phi, mus, sigs):
    k = len(phi)
    total = 0.0
    for xi in X:
        px = sum(phi[j] * multivariate_normal(mus[j], sigs[j]).pdf(xi) for j in range(k))
        total += np.log(px + 1e-300)
    return total

# Show how log-likelihood varies as we move mu_1 along a line
mu1_x_range = np.linspace(-2, 4, 60)
ll_vals = []
for mx in mu1_x_range:
    mus_try = mus.copy(); mus_try[0, 0] = mx
    ll_vals.append(gmm_log_likelihood(X[:50], phi, mus_try, sigs))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(mu1_x_range, ll_vals, 'b-', lw=2.5)
ax.axvline(mus[0, 0], color='green', ls='--', lw=2,
           label=f'True $\\mu_{{1,x}} = {mus[0,0]:.1f}$')
ax.axvline(mu1_x_range[np.argmax(ll_vals)], color='red', ls=':', lw=2,
           label=f'Empirical max at $\\mu_{{1,x}} \\approx {mu1_x_range[np.argmax(ll_vals)]:.2f}$')
ax.set_xlabel('$\\mu_{1,x}$ (x-component of first centroid)')
ax.set_ylabel('Log-likelihood $\\ell$')
ax.set_title('Marginal log-likelihood as a function of $\\mu_{1,x}$\n'
             'Cannot be solved in closed form when $z^{(i)}$ unobserved → need EM')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 4. Derivation Pathway

### Derivation pathway

In [ ]:
from IPython.display import HTML
HTML("""
<svg xmlns="http://www.w3.org/2000/svg" width="780" height="374"
     viewBox="0 0 780 374" font-family="Segoe UI, Arial, sans-serif">
  <defs>
    <marker id="ah" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#444"/>
    </marker>
    <marker id="ahd" markerWidth="10" markerHeight="7" refX="9" refY="3.5"
            orient="auto" markerUnits="userSpaceOnUse">
      <polygon points="0 0,10 3.5,0 7" fill="#999"/>
    </marker>
  </defs>

  <!-- Row 1: Generative model -->
  <rect x="10" y="12" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="35" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Generative model</text>
  <text x="102" y="52" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">z&#x2071; ~ Multi(&#x3c6;), x|z ~ N</text>
  <line x1="197" y1="35" x2="216" y2="35"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="12" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="40" font-size="13" text-anchor="middle" fill="#333"
        >latent z&#x2071; unobserved &#x2014; parameters &#x3c6;, {&#x3bc;&#x2c7;}, {&#x3a3;&#x2c7;} to estimate</text>

  <!-- step 1&#x2192;2 -->
  <line x1="102" y1="58" x2="102" y2="108"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="82" font-size="11.5" font-style="italic" fill="#555"
        >marginalise over z: p(x&#x2071;) = &#x2211;&#x2c7; &#x3c6;&#x2c7; N(x&#x2071;|&#x3bc;&#x2c7;,&#x3a3;&#x2c7;)</text>

  <!-- Row 2: Marginal likelihood -->
  <rect x="10" y="112" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="135" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">Marginal</text>
  <text x="102" y="152" font-size="12.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">log-likelihood</text>
  <line x1="197" y1="135" x2="216" y2="135"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="112" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="140" font-size="13" text-anchor="middle" fill="#333"
        >&#x2113; = &#x2211;&#x2071; log &#x2211;&#x2c7; &#x3c6;&#x2c7; N(x&#x2071;|&#x3bc;&#x2c7;,&#x3a3;&#x2c7;) &#x2014; log-sum has no closed-form gradient zero</text>

  <!-- step 2&#x2192;3 -->
  <line x1="102" y1="158" x2="102" y2="208"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>
  <text x="114" y="182" font-size="11.5" font-style="italic" fill="#555"
        >need EM to optimise &#x2113; &#x2014; covered in next notebook</text>

  <!-- Row 3: EM needed -->
  <rect x="10" y="212" width="185" height="46" rx="7"
        fill="#ffffff" stroke="#2166ac" stroke-width="2"/>
  <text x="102" y="240" font-size="13.5" font-weight="600"
        text-anchor="middle" fill="#2166ac">EM algorithm</text>
  <line x1="197" y1="235" x2="216" y2="235"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="212" width="548" height="46" rx="7"
        fill="#eef2f7" stroke="#b0bec5" stroke-width="1.5"/>
  <text x="495" y="240" font-size="13" text-anchor="middle" fill="#333"
        >E-step: compute soft assignments w&#x2c7;&#x2071; = p(z&#x2071;=j|x&#x2071;); M-step: closed-form update</text>

  <!-- step 3&#x2192;4 -->
  <line x1="102" y1="258" x2="102" y2="308"
        stroke="#999" stroke-width="1.8" stroke-dasharray="5,3"
        marker-end="url(#ahd)"/>

  <!-- Row 4: Fitted GMM -->
  <rect x="10" y="312" width="185" height="46" rx="7"
        fill="#1a5fa8" stroke="#1a5fa8" stroke-width="2"/>
  <text x="102" y="340" font-size="13.5" font-weight="700"
        text-anchor="middle" fill="#ffffff">Fitted GMM</text>
  <line x1="197" y1="335" x2="216" y2="335"
        stroke="#444" stroke-width="2" marker-end="url(#ah)"/>
  <rect x="221" y="312" width="548" height="46" rx="7"
        fill="#dce8f8" stroke="#7aadd4" stroke-width="1.5"/>
  <text x="495" y="440" font-size="13" text-anchor="middle" fill="#333"
        >soft cluster assignments &#x2014; generalises k-means (hard assignments) to probabilistic model</text>
</svg>
""")

## Summary

| Concept | Formula | Key Insight |
|---|---|---|
| GMM generative model | $z^{(i)} \sim \text{Multi}(\phi)$, $x^{(i)}\lvert z^{(i)}=j \sim \mathcal{N}(\mu_j, \Sigma_j)$ | Mixture components correspond to latent clusters |
| Marginal density | $p(x^{(i)}) = \sum_j \phi_j \mathcal{N}(x^{(i)};\mu_j,\Sigma_j)$ | Summing out the latent variable |
| Marginal log-likelihood | $\ell = \sum_i \log \sum_j \phi_j \mathcal{N}(x^{(i)};\mu_j,\Sigma_j)$ | Log-of-sum has no closed-form optimum |
| MLE with known $z$ | $\hat{\phi}_j = \frac{1}{m}\sum_i \mathbf{1}\{z^{(i)}=j\}$, etc. | Reduces to Gaussian discriminant analysis |
| GMM vs. k-means | Soft assignments $w_j^{(i)} = p(z^{(i)}=j\lvert x^{(i)})$ | k-means is the hard-assignment limiting case |

**Key insight:** the mixture of Gaussians is a natural probabilistic extension of k-means that uses soft cluster assignments; but the marginal likelihood has no closed-form MLE because the latent $z$ couples all parameters — this motivates the EM algorithm.